# DataView: Exploração e Análise de Dados de Vendas

Mini-Projeto Avaliativo - Módulo 1 - Semana 08
Disciplina: Fundamentos de Dados, Programação e Análise Preditiva com Python

**Analista de Dados Júnior** simulando um relatório de vendas para a diretoria,
cobrindo todo o pipeline: carga, inspeção, limpeza, outliers, colunas derivadas,
métricas agregadas, segmentação de clientes, estatísticas com NumPy, visualizações,
funções reutilizáveis e exportação de relatórios (CSV/JSON).

---

### Contexto do Projeto

Este projeto simula o cenário de uma **empresa de varejo de vestuário e moda** que
apresentará um relatório trimestral de vendas à diretoria executiva. O dataset
contém **15.000 registros** com **16 colunas**, abrangendo janeiro a dezembro de 2024,
com dados de clientes, produtos (20 itens em 5 categorias de vestuário), canais de
venda (loja física, e-commerce, app), regiões geográficas, formas de pagamento,
descontos e frete.


### Dados Intencionalmente Sujos
Para simular cenários reais e praticar técnicas de limpeza, o dataset contém
**dados sujos propositalmente inseridos**: valores nulos (~5% em quantidade/preço),
datas inválidas (~2%), strings com espaços extras (~3%), e outliers de quantidade
(~3% com valores entre 30 e 80 unidades). A RF03 e RF04 tratam esses problemas.


## Passo 1 — Importação de Bibliotecas

Nesta etapa inicial, importamos todas as bibliotecas necessárias para o pipeline
completo de análise de dados. Cada biblioteca desempenha um papel específico:

| Biblioteca | Papel no Projeto |
|---|---|
| **pandas** | Manipulação e análise de dados via DataFrames — estrutura principal do projeto |
| **numpy** | Operações numéricas vetorizadas, cálculos estatísticos, broadcasting e arrays |
| **random** | Geração de dados sintéticos aleatórios para o dataset |
| **os** | Criação de diretórios e verificação de existência de arquivos |
| **re** | Expressões regulares para limpeza e normalização de strings |
| **json** | Leitura e escrita de arquivos no formato JSON |
| **matplotlib** | Criação de gráficos (linha, barras) e exportação em PNG |
| **seaborn** | Gráficos estatísticos avançados (boxplot) com tema profissional |
| **datetime** | Manipulação de datas, extração de componentes temporais e cálculo de intervalos |


In [31]:
# Biblioteca principal para manipulação e análise de dados via DataFrames
import pandas as pd
# Biblioteca para operações numéricas vetorizadas, arrays e estatísticas
import numpy as np
# Biblioteca para geração de números aleatórios (dados sintéticos)
import random
# Biblioteca do sistema operacional para criação de diretórios e verificação de arquivos
import os
# Biblioteca para expressões regulares — limpeza e normalização de strings
import re
# Biblioteca para leitura e escrita de arquivos no formato JSON
import json
# Biblioteca para criação de gráficos (matplotlib) e exportação em PNG
import matplotlib.pyplot as plt
# Biblioteca para gráficos estatísticos avançados (boxplot, theme profissional)
import seaborn as sns
# Biblioteca para manipulação de datas e extração de componentes temporais
from datetime import datetime, timedelta

## RF01 — Criar ou Carregar o Dataset de Vendas

Geramos o dataset sinteticamente, com sujeira intencional (nulos, datas inválidas,
strings com espaços extras) para praticar a limpeza na RF03.

### Sobre o Dataset

O dataset representa uma empresa de **varejo de vestuário e moda** com operações
em todo o Brasil. Ele contém **15.000 registros** e **16 colunas**, abrangendo
o período de janeiro a dezembro de 2024.

**Colunas do dataset:** id_venda, data_venda, cliente, produto, categoria,
tamanho, cor, regiao, canal_venda, forma_pagamento, parcelas, quantidade,
preco_unitario, custo_unitario, percentual_desconto, valor_frete.

**20 produtos** distribuídos em **5 categorias** de vestuário:
- Moda Masculina (4 itens)
- Moda Feminina (4 itens)
- Calçados (4 itens)
- Moda Íntima e Praia (4 itens)
- Acessórios de Moda (4 itens)

**Sujeira intencional inserida no dataset:**
- Valores nulos em quantidade e preço (~5% dos registros)
- Datas inválidas como string "DATA INVALIDA" (~2%)
- Strings com espaços extras no nome dos produtos (~3%)
- Outliers de quantidade: vendas com 30 a 80 unidades (~3%)

### Por que dados sintéticos?

Utilizar dados gerados sinteticamente garante **reprodutibilidade total** — qualquer
pessoa que execute o notebook com a mesma seed (42) obterá exatamente os mesmos
resultados. Além disso, permite controlar a proporção e o tipo de "sujeira" inserida,
facilitando o treinamento em técnicas de limpeza de dados.

In [32]:
def gerar_dataset_vendas_vestuario(n_registros=15000, n_clientes=250, seed=42):
    """
    Gera um dataset sintético de vendas focado no segmento de vestuário e moda,
    incluindo intencionalmente dados sujos como valores nulos, strings com espaços,
    datas corrompidas e outliers de quantidade para testar o pipeline do projeto.
    Incorpora regras dinâmicas de cobrança de frete baseadas no canal de venda.
    """
    # Fixa a seed para reprodutibilidade — sempre os mesmos dados aleatórios
    random.seed(seed)
    np.random.seed(seed)

    # 1. Portfólio de 20 produtos de vestuário com preços em reais
    # Cobrem 5 categorias: Moda Masculina, Moda Feminina, Calçados, Moda Íntima/Praia, Acessórios
    precos = {
        # Categoria: Roupas Masculinas
        'Camisa Polo Masculina': 129.90, 'Calça Jeans Casual': 189.90, 'Camiseta Básica Algodão': 49.90, 'Bermuda Sarja': 99.90,
        # Categoria: Roupas Femininas
        'Vestido Estampado Midi': 199.90, 'Blusa Crepe Feminina': 79.90, 'Calça Alfaiataria': 159.90, 'Saia Plissada': 119.90,
        # Categoria: Calçados
        'Ténis Desportivo Run': 349.90, 'Sapato Social Couro': 289.90, 'Sandália Salto Bloco': 149.90, 'Chinelo Slim Conforto': 39.90,
        # Categoria: Moda Íntima e Praia
        'Kit Cuecas Boxer 3un': 79.90, 'Conjunto Lingerie Renda': 119.90, 'Biquíni Cortininha': 139.90, 'Sunga de Praia': 69.90,
        # Categoria: Acessórios de Moda
        'Óculos de Sol Aviador': 179.90, 'Cinto de Couro': 69.90, 'Mochila Urbana': 219.90, 'Boné Ajustável Aba Curva': 59.90
    }

    # 2. Mapeamento de cada produto para sua categoria (5 categorias no total)
    categorias = {
        'Camisa Polo Masculina': 'Moda Masculina', 'Calça Jeans Casual': 'Moda Masculina', 'Camiseta Básica Algodão': 'Moda Masculina', 'Bermuda Sarja': 'Moda Masculina',
        'Vestido Estampado Midi': 'Moda Feminina', 'Blusa Crepe Feminina': 'Moda Feminina', 'Calça Alfaiataria': 'Moda Feminina', 'Saia Plissada': 'Moda Feminina',
        'Ténis Desportivo Run': 'Calçados', 'Sapato Social Couro': 'Calçados', 'Sandália Salto Bloco': 'Calçados', 'Chinelo Slim Conforto': 'Calçados',
        'Kit Cuecas Boxer 3un': 'Moda Íntima e Praia', 'Conjunto Lingerie Renda': 'Moda Íntima e Praia', 'Biquíni Cortininha': 'Moda Íntima e Praia', 'Sunga de Praia': 'Moda Íntima e Praia',
        'Óculos de Sol Aviador': 'Acessórios', 'Cinto de Couro': 'Acessórios', 'Mochila Urbana': 'Acessórios', 'Boné Ajustável Aba Curva': 'Acessórios'
    }

    # Listas auxiliares para geração aleatória
    produtos = list(precos.keys())
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    # Gera 250 clientes com nomes padronizados (Cliente_0001 a Cliente_0250)
    clientes = [f"Cliente_{i:04d}" for i in range(1, n_clientes + 1)]

    # Data inicial do período de análise (jan/2024)
    data_inicio = datetime(2024, 1, 1)
    dados = []  # Lista que acumulará os registros gerados
    
    # Loop principal: gera 15.000 registros de venda
    for i in range(n_registros):
        produto = random.choice(produtos)  # Seleciona produto aleatório do portfólio
        quantidade = random.randint(1, 3)  # Quantidade padrão: 1 a 3 unidades
        preco = precos[produto]  # Busca o preço unitário do produto
        data = data_inicio + timedelta(days=random.randint(0, 364))  # Data aleatória em 2024

        # --- REGRAS DE SUJEIRA INTENCIONAL PARA O PIPELINE DE LIMPEZA ---
        # ~5% das vendas ficam sem quantidade (nulo)
        if random.random() < 0.05: quantidade = None
        # ~4% das vendas ficam sem preço (nulo)
        if random.random() < 0.04: preco = None
        # ~3% dos produtos têm espaços extras no nome (para testar regex)
        if random.random() < 0.03: produto = "    " + produto + "  "
        # ~3% das vendas têm outliers de quantidade (30 a 80 unidades)
        if random.random() < 0.03: quantidade = random.randint(30, 80)
        # ~2% das datas são inválidas (string 'DATA INVALIDA')
        data_str = data.strftime("%Y-%m-%d") if random.random() > 0.02 else "DATA INVALIDA"

        # Custo unitário = 45% do preço original (margem de 55%)
        custo = round(precos[produto.strip()] * 0.45, 2) if preco is not None else None
        
        # Sorteia tamanho baseado no tipo de produto
        # Calçados usam números (36-44), roupas usam letras (P, M, G, GG)
        categoria_prod = categorias.get(produto.strip(), "Outros")
        tamanho = random.choice(['36', '38', '40', '42', '44']) if categoria_prod == 'Calçados' else random.choice(['P', 'M', 'G', 'GG'])
        
        # Sorteia cor de entre 5 opções disponíveis
        cor = random.choice(['Preto', 'Branco', 'Azul', 'Vermelho', 'Nude'])
        
        # Lógica de desconto: 15% das vendas recebem desconto de 5% a 30%
        desconto = round(random.uniform(0.05, 0.30), 2) if random.random() < 0.15 else 0.0
        
        # Forma de pagamento: Cartão permite parcelamento (1-12x), Pix e Boleto são à vista
        forma_pgto = random.choice(['Cartão de Crédito', 'Pix', 'Boleto'])
        parcelas = random.randint(1, 12) if forma_pgto == 'Cartão de Crédito' else 1
        
        # Canal de venda: loja física, e-commerce ou app mobile
        canal_venda = random.choice(['E-commerce', 'App', 'Loja Física'])

        # Lógica de frete: gratuito para loja física, pago (R$20-R$100) para canais online
        if canal_venda in ['App', 'E-commerce']:
            valor_frete = round(random.uniform(20.00, 100.00), 2)
        else:
            valor_frete = 0.00

        # Monta o dicionário do registro de venda com todas as 16 colunas
        dados.append({
            "id_venda": i + 1,
            "data_venda": data_str,
            "cliente": random.choice(clientes),
            "produto": produto,
            "categoria": categoria_prod,
            "tamanho": tamanho,
            "cor": cor,
            "regiao": random.choice(regioes),
            "canal_venda": canal_venda, 
            "forma_pagamento": forma_pgto, 
            "parcelas": parcelas,
            "quantidade": quantidade,
            "preco_unitario": preco,
            "custo_unitario": custo, 
            "percentual_desconto": desconto,
            "valor_frete": valor_frete  # Frete: 0 para loja física, variável para online
        })

    # Converte a lista de dicionários em DataFrame do pandas
    return pd.DataFrame(dados)

# Gera o dataset com 15.000 registros e 250 clientes
df_bruto = gerar_dataset_vendas_vestuario(n_registros=15000, n_clientes=250, seed=42)
# Cria o diretório 'data/raw' se não existir
os.makedirs('../data/raw', exist_ok=True)
# Salva o dataset bruto em CSV para referência futura
df_bruto.to_csv("../data/raw/vendas.csv", index=False)

print(f"Dataset de vestuário gerado com {len(df_bruto):,} registros.")
df_bruto.head()  # Exibe as 5 primeiras linhas para verificação visual


Dataset de vestuário gerado com 15,000 registros.


,id_venda,data_venda,cliente,produto,categoria,tamanho,cor,regiao,canal_venda,forma_pagamento,parcelas,quantidade,preco_unitario,custo_unitario,percentual_desconto,valor_frete
0,1,2024-05-20,Cliente_0007,Bermuda Sarja,Moda Masculina,GG,Preto,Norte,Loja Física,Boleto,1,1.0,99.9,44.96,0.1,0.00
1,2,2024-11-28,Cliente_0098,Calça Alfaiataria,Moda Feminina,M,Vermelho,Sudeste,App,Cartão de Crédito,4,3.0,159.9,71.95,0.0,28.18
2,3,2024-11-05,Cliente_0227,Chinelo Slim Conforto,Calçados,36,Nude,Nordeste,Loja Física,Boleto,1,2.0,39.9,17.95,0.0,0.00
3,4,2024-12-26,Cliente_0091,Mochila Urbana,Acessórios,P,Vermelho,Sul,App,Boleto,1,1.0,219.9,98.95,0.0,33.01
4,5,2024-12-15,Cliente_0143,Ténis Desportivo Run,Calçados,42,Azul,Sul,Loja Física,Boleto,1,3.0,349.9,157.45,0.0,0.00


## RF02 — Inspecionar e Descrever os Dados

Antes de qualquer tratamento, é essencial **entender a estrutura, os tipos de dados
e a qualidade do dataset bruto**. A inspeção inicial identifica:

- **Dimensões do dataset** (shape): número de linhas e colunas
- **Tipos de dados** (dtypes): quais colunas são numéricas, categóricas ou de texto
- **Valores nulos** (isnull().sum()): colunas com dados faltantes que precisarão de tratamento
- **Primeiros registros** (head): amostra visual para verificar formatação e conteúdo
- **Estatísticas descritivas** (describe): média, mediana, desvio padrão, mínimo, máximo e quartis

Nesta etapa, também utilizamos um **laço `while`** para demonstrar a verificação
de qualidade de forma iterativa, verificando nulos e duplicatas em colunas críticas
(quantidade, preco_unitario, data_venda, produto).


In [33]:
def inspecionar_dados(df):
    """Exibe informações básicas do DataFrame."""
    # Cabeçalho da inspeção
    print("\n=== INSPEÇÃO INICIAL DO DATASET ===")
    # Shape: tupla (linhas, colunas) — ex: (15000, 16)
    print(f"Shape: {df.shape}")
    # Lista de todas as colunas do dataset
    print(f"\nColunas: {list(df.columns)}")
    # Tipos de dados: identifica int, float, object (string) — importante para limpeza
    print(f"\nTipos de dados:\n{df.dtypes}")
    # Contagem de valores nulos por coluna — indica quais colunas precisam de tratamento
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    # Primeiras 5 linhas — verificação visual de formatação e conteúdo
    print(f"\nPrimeiros registros:\n{df.head()}")
    # Estatísticas descritivas: média, desvio, min, max, quartis para colunas numéricas
    print(f"\nEstatísticas descritivas:\n{df.describe()}")
    # describe(include='all') inclui estatísticas de colunas categóricas (top, freq)
    return df.describe(include="all")

# Executa a inspeção no dataset bruto e armazena o resumo
resumo = inspecionar_dados(df_bruto)

# --- Demonstra while: verificação iterativa de qualidade nas colunas críticas ---
print("\n=== VERIFICAÇÃO DE QUALIDADE (while) ===")
# Colunas que são críticas para o negócio — precisam de verificação obrigatória
colunas_check = ["quantidade", "preco_unitario", "data_venda", "produto"]
idx = 0  # Índice inicial do loop while
while idx < len(colunas_check):  # Itera até verificar todas as colunas da lista
    col = colunas_check[idx]  # Seleciona a coluna atual
    n_nulos = int(df_bruto[col].isnull().sum())  # Conta valores nulos na coluna
    n_duplicados = int(df_bruto[col].duplicated().sum())  # Conta valores duplicados
    # Status: OK se não houver nulos, senão mostra a quantidade com alerta
    status = "OK" if n_nulos == 0 else f"⚠ {n_nulos} nulos"
    print(f"  {col}: {status} | {n_duplicados} duplicados")
    idx += 1  # Avança para a próxima coluna



=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (15000, 16)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'tamanho', 'cor', 'regiao', 'canal_venda', 'forma_pagamento', 'parcelas', 'quantidade', 'preco_unitario', 'custo_unitario', 'percentual_desconto', 'valor_frete']

Tipos de dados:
id_venda                 int64
data_venda                 str
cliente                    str
produto                    str
categoria                  str
tamanho                    str
cor                        str
regiao                     str
canal_venda                str
forma_pagamento            str
parcelas                 int64
quantidade             float64
preco_unitario         float64
custo_unitario         float64
percentual_desconto    float64
valor_frete            float64
dtype: object

Valores nulos por coluna:
id_venda                 0
data_venda               0
cliente                  0
produto                  0
categoria                0
tamanho              

## RF03 — Limpar e Tratar os Dados

Tratamos valores nulos em colunas críticas, datas inválidas e strings com espaços
extras (via expressões regulares), registrando um relatório do impacto de cada etapa.

### Pipeline de Limpeza em 4 Etapas

1. **Normalização de strings com regex** — A função `limpar_strings_regex` utiliza a
   expressão regular `r"\s+"` para colapsar múltiplos espaços internos em um único
   espaço e remove espaços nas pontas com `.strip()`. Preserva células nulas sem erro.

2. **Conversão de datas** — A coluna `data_venda` é convertida para o tipo datetime
   usando `pd.to_datetime(errors="coerce")`. Valores inválidos (como "DATA INVALIDA")
   são automaticamente convertidos para `NaT` e随后 removidos com `dropna`.

3. **Remoção de nulos em colunas críticas** — Registros com valores nulos em
   `quantidade` ou `preco_unitario` são removidos. Optamos por **remover** em vez de
   imputar porque vendas sem quantidade ou preço não contribuem para métricas de
   receita e ticket médio — imputar esses valores introduziria viés nos resultados.

4. **Garantia de tipos numéricos** — As colunas `quantidade` e `preco_unitario` são
   convertidas para `int` e `float`, respectivamente, garantindo operações matemáticas
   corretas nas etapas subsequentes.

### Relatório de Auditoria

Após cada etapa de limpeza, um relatório detalhado é gerado mostrando o número
absoluto e percentual de registros removidos, permitindo transparência e rastreabilidade
do impacto de cada tratamento na volumetria do dataset.


In [34]:
def limpar_strings_regex(df, colunas):
    """
    Usa expressões regulares para normalizar colunas de texto:
    - Colapsa múltiplos espaços internos em um único espaço (re.sub)
    - Remove espaços nas pontas da string (.strip())
    - Preserva células nulas sem lançar erro (pd.notna)
    """
    df = df.copy()  # Cópia para não modificar o DataFrame original
    for col in colunas:  # Itera sobre cada coluna de texto identificada
        # Aplica regex r"\s+" → substitui 1+ espaços/tabs por 1 espaço; strip() removes bordas
        df[col] = df[col].apply(
            lambda s: re.sub(r"\s+", " ", str(s)).strip() if pd.notna(s) else s
        )
    return df


def exibir_relatorio_auditoria(relatorio):
    """
    Formata, alinha e renderiza o relatório final de auditoria com o cálculo
    de representação percentual para cada canal de exclusão do pipeline.
    """
    base_inicial = relatorio["registros_iniciais"]  # Total de registros do dataset bruto
    
    # Cálculo das proporções percentuais: (nº removidos / total inicial) × 100
    pct_nulas_qtd = (relatorio["nulos_quantidade"] / base_inicial) * 100
    pct_nulas_prc = (relatorio["nulos_preco_unitario"] / base_inicial) * 100
    pct_subtotal_nulos = (relatorio["linhas_nulas_removidas"] / base_inicial) * 100
    pct_datas = (relatorio["datas_invalidas_removidas"] / base_inicial) * 100
    pct_removidos = (relatorio["registros_removidos_total"] / base_inicial) * 100
    pct_finais = (relatorio["registros_finais"] / base_inicial) * 100

    # Formatação do relatório com alinhamento e separadores visuais
    print("=" * 72)
    print(f"{'=== RELATÓRIO DE LIMPEZA E AUDITORIA ESTATÍSTICA ===':^72}")
    print("=" * 72)
    print(f" -> Registros Iniciais no Dataset:  {base_inicial:<8} [100.00%]")
    print("\n DETALHAMENTO DE EXCLUSÕES POR CATEGORIA/COLUNA:")
    print("-" * 72)
    # Seção de nulos: mostra contagem e percentual por coluna
    print(" [Campos Nulos Detectados]")
    print(f"  • quantidade:                     {relatorio['nulos_quantidade']:<8} [{pct_nulas_qtd:>6.2f}%]")
    print(f"  • preco_unitario:                 {relatorio['nulos_preco_unitario']:<8} [{pct_nulas_prc:>6.2f}%]")
    print(f"   Subtotal de Linhas Nulas:        {relatorio['linhas_nulas_removidas']:<8} [{pct_subtotal_nulos:>6.2f}%]")
    # Seção de datas inválidas: contagem e percentual
    print("\n [Inconsistências Cronológicas]")
    print(f"  • data_venda ('DATA INVALIDA'):   {relatorio['datas_invalidas_removidas']:<8} [{pct_datas:>6.2f}%]")
    print(f"   Subtotal de Datas Inválidas:     {relatorio['datas_invalidas_removidas']:<8} [{pct_datas:>6.2f}%]")
    print("-" * 72)
    # Resumo: total removido vs. registros finais (base limpa)
    print(f" -> Total de Registros Removidos:   {relatorio['registros_removidos_total']:<8} [{pct_removidos:>6.2f}%]")
    print(f" -> Registros Finais (Base Limpa):  {relatorio['registros_finais']:<8} [{pct_finais:>6.2f}%]")
    print("=" * 72)


def limpar_dados(df):
    """
    Limpa o DataFrame de vendas em quatro etapas:
    1. Normaliza strings com regex (espaços extras)
    2. Converte datas e remove registros com datas inválidas
    3. Remove linhas com valores nulos em colunas obrigatórias (mapeando a origem)
    4. Garante os tipos numéricos corretos
    Retorna: (df_limpo, relatorio)
    """
    df = df.copy()  # Cópia para preservar o DataFrame original
    n_inicial = len(df)  # Guarda o total inicial para cálculo de impacto
    relatorio = {"registros_iniciais": n_inicial}  # Dicionário de metadados do relatório

    # --- Etapa 1: limpeza de strings com regex ---
    # Identifica colunas de tipo string/object automaticamente
    colunas_texto = df.select_dtypes(include="string").columns
    df = limpar_strings_regex(df, colunas_texto)

    # --- Etapa 2: conversão de datas ---
    # pd.to_datetime com errors='coerce' converte datas inválidas para NaT
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
    relatorio["datas_invalidas_removidas"] = int(df["data_venda"].isnull().sum())
    # Remove linhas com datas nulas/inválidas
    df = df.dropna(subset=["data_venda"])

    # --- Etapa 3: contagem e remoção de nulos por coluna ---
    # Conta nulos individuais ANTES do dropna para o relatório
    relatorio["nulos_quantidade"] = int(df["quantidade"].isnull().sum())
    relatorio["nulos_preco_unitario"] = int(df["preco_unitario"].isnull().sum())
    
    n_antes_nulos = len(df)  # Guarda total antes da remoção de nulos
    # Remove linhas com nulos em colunas críticas para cálculos de receita
    df = df.dropna(subset=["quantidade", "preco_unitario"])
    # Calcula quantas linhas foram removidas nesta etapa
    relatorio["linhas_nulas_removidas"] = n_antes_nulos - len(df)

    # --- Etapa 4: garantia de tipos numéricos ---
    # Converte quantidade para int e preço para float para operações matemáticas
    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)

    # Consolidação dos metadados volumétricos finais no relatório
    relatorio["registros_finais"] = len(df)
    relatorio["registros_removidos_total"] = n_inicial - len(df)

    return df, relatorio  # Retorna DataFrame limpo e dicionário de relatório


# =========================================================================
# EXECUÇÃO DO PIPELINE DE LIMPEZA
# =========================================================================
# Executa a limpeza estrutural completa e gera a tabela estável v1
df_v1, relatorio = limpar_dados(df_bruto)

# Exibe o relatório formatado de auditoria
exibir_relatorio_auditoria(relatorio)


# Salva a versão limpa (v1) com outliers ainda mantidos
os.makedirs("../data/processed/v1_com_outliers", exist_ok=True)
df_v1.to_csv("../data/processed/v1_com_outliers/vendas_v1.csv", index=False)
print("\n[OK] Versão v1 gravada com sucesso em: data/processed/v1_com_outliers/vendas_v1.csv\n")
df_v1.head()


          === RELATÓRIO DE LIMPEZA E AUDITORIA ESTATÍSTICA ===          
 -> Registros Iniciais no Dataset:  15000    [100.00%]

 DETALHAMENTO DE EXCLUSÕES POR CATEGORIA/COLUNA:
------------------------------------------------------------------------
 [Campos Nulos Detectados]
  • quantidade:                     750      [  5.00%]
  • preco_unitario:                 590      [  3.93%]
   Subtotal de Linhas Nulas:        1307     [  8.71%]

 [Inconsistências Cronológicas]
  • data_venda ('DATA INVALIDA'):   291      [  1.94%]
   Subtotal de Datas Inválidas:     291      [  1.94%]
------------------------------------------------------------------------
 -> Total de Registros Removidos:   1598     [ 10.65%]
 -> Registros Finais (Base Limpa):  13402    [ 89.35%]

[OK] Versão v1 gravada com sucesso em: data/processed/v1_com_outliers/vendas_v1.csv



,id_venda,data_venda,cliente,produto,categoria,tamanho,cor,regiao,canal_venda,forma_pagamento,parcelas,quantidade,preco_unitario,custo_unitario,percentual_desconto,valor_frete
0,1,2024-05-20,Cliente_0007,Bermuda Sarja,Moda Masculina,GG,Preto,Norte,Loja Física,Boleto,1,1,99.9,44.96,0.1,0.00
1,2,2024-11-28,Cliente_0098,Calça Alfaiataria,Moda Feminina,M,Vermelho,Sudeste,App,Cartão de Crédito,4,3,159.9,71.95,0.0,28.18
2,3,2024-11-05,Cliente_0227,Chinelo Slim Conforto,Calçados,36,Nude,Nordeste,Loja Física,Boleto,1,2,39.9,17.95,0.0,0.00
3,4,2024-12-26,Cliente_0091,Mochila Urbana,Acessórios,P,Vermelho,Sul,App,Boleto,1,1,219.9,98.95,0.0,33.01
4,5,2024-12-15,Cliente_0143,Ténis Desportivo Run,Calçados,42,Azul,Sul,Loja Física,Boleto,1,3,349.9,157.45,0.0,0.00


## RF04 — Detectar e Tratar Outliers (versões v1 e v2)

Detectamos outliers em `quantidade` e `receita_total` pelo método IQR. A v1 mantém
os outliers; a v2 reaproveita a mesma função de limpeza/tratamento para gerar uma
base sem outliers.

### O que são Outliers?

Outliers são valores que se **desviam significativamente** do padrão esperado nos dados.
No contexto de vendas, uma venda com 80 unidades de um produto pode ser um erro de
digitação ou uma anomalia que distorce métricas como ticket médio e receita total.

### Método IQR (Intervalo Interquartil)

O método IQR é **robusto** porque não assume que os dados seguem uma distribuição
normal. A fórmula é:

- **Q1** = percentil 25 (25% dos valores estão abaixo dele)
- **Q3** = percentil 75 (75% dos valores estão abaixo dele)
- **IQR** = Q3 - Q1
- **Limite Inferior** = Q1 - 1.5 × IQR
- **Limite Superior** = Q3 + 1.5 × IQR

Valores fora desses limites são considerados outliers.

### Por que verificar `quantidade` e `receita_total`?

- **quantidade**: captura outliers em vendas com quantidades individuais anormais
- **receita_total**: captura o efeito combinado de preço × quantidade, identificando
  vendas com receita total muito acima do padrão mesmo quando a quantidade é normal

### Duas Versões

- **v1 (df_v1)**: dados limpos mas **com outliers mantidos** — útil para análise exploratória
- **v2 (df_v2)**: dados limpos **sem outliers** — base oficial para métricas agregadas


### Stripplot — Visualização dos Outliers (Antes e Depois)

O **stripplot** mostra cada registro como um ponto individual, permitindo visualizar
a distribuição real dos dados e identificar os outliers (pontos distantes da concentração principal).
Exibimos o gráfico **antes** e **após** a remoção para comparar o efeito do tratamento.

In [ ]:
def remover_outliers(df, colunas, fator=1.5):
    """Remove outliers usando IQR. Retorna df sem outliers e dict com estatisticas."""
    df = df.copy()
    info = {}
    for col in colunas:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lim_inf = q1 - fator * iqr
        lim_sup = q3 + fator * iqr
        mask = (df[col] < lim_inf) | (df[col] > lim_sup)
        info[col] = {'Q1': q1, 'Q3': q3, 'IQR': iqr,
                      'lim_inf': lim_inf, 'lim_sup': lim_sup,
                      'n_outliers': int(mask.sum()),
                      'pct': round(mask.sum() / len(df) * 100, 2)}
        pct = info[col]['pct']
        print(f'  {col}: {mask.sum()} outliers ({pct}%) | '
              f'lim_inf={lim_inf:.2f}  lim_sup={lim_sup:.2f}')
        df = df[~mask]
    return df, info


def plot_stripplot(df, colunas, titulo='Stripplot', caminho=None):
    """Exibe stripplot para cada coluna. Se caminho informado, salva em PNG."""
    n = len(colunas)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]
    palette = sns.color_palette('Set2', n)
    for ax, col, color in zip(axes, colunas, palette):
        sns.stripplot(data=df, y=col, ax=ax, color=color,
                      alpha=0.4, size=4, jitter=True)
        ax.set_title(col.replace('_', ' ').title(), fontsize=12, fontweight='bold')
        ax.set_ylabel(col)
    plt.suptitle(titulo, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    if caminho:
        plt.savefig(caminho, dpi=150, bbox_inches='tight')
        print(f'  Salvo: {caminho}')
    plt.show()


# Cria pasta de saida
os.makedirs('../outputs/graficos', exist_ok=True)

# Prepara dados temporarios com receita_total
df_v1_tmp = df_v1.copy()
df_v1_tmp["receita_total"] = df_v1_tmp["quantidade"] * df_v1_tmp["preco_unitario"]
colunas_analise = ["quantidade", "receita_total"]

# Stripplot ANTES da remocao
plot_stripplot(df_v1_tmp, colunas_analise,
               titulo='Stripplot (Antes da Remocao)',
               caminho='../outputs/graficos/stripplot_antes.png')

# Remove outliers
df_v2, info_outliers = remover_outliers(df_v1_tmp, colunas_analise)

# Stripplot DEPOIS da remocao
plot_stripplot(df_v2, colunas_analise,
               titulo='Stripplot (Depois da Remocao)',
               caminho='../outputs/graficos/stripplot_depois.png')

# Remove receita_total temporaria (sera recriada no RF05)
df_v2 = df_v2.drop(columns=["receita_total"])

# Resumo
print(f'\nv1 = {len(df_v1_tmp)} linhas (com outliers)')
print(f'v2 = {len(df_v2)} linhas (outliers removidos)')
print(f'Diferenca = {len(df_v1_tmp) - len(df_v2)} linhas removidas')

# Salva v2
os.makedirs('../data/processed/v2_outliers_tratado', exist_ok=True)
df_v2.to_csv('../data/processed/v2_outliers_tratado/vendas_v2.csv', index=False)
print('\nv2 salvo em data/processed/v2_outliers_tratado/')


## RF05 — Criar Colunas Derivadas com Transformações

A partir daqui usamos `df_v2` (outliers tratados) como base oficial da análise.

### O que são Colunas Derivadas?

Colunas derivadas são **novas colunas criadas a partir de dados existentes** no
dataset. Elas enriquecem a análise com informações calculadas que facilitam
agrupamentos, filtros e visualizações.

### 5 Colunas Criadas nesta Etapa

| Coluna | Descrição | Cálculo |
|---|---|---|
| `receita_total` | Valor total da linha de venda | `quantidade × preco_unitario` |
| `mes` | Mês numérico da venda | `.dt.month` |
| `trimestre` | Trimestre formatado (Q1-Q4) | `.dt.quarter` + `lambda` |
| `ano` | Ano da venda | `.dt.year` |
| `faixa_receita_item` | Classificação do valor da venda | `np.select` |

### Uso do `np.select`

A função `np.select` é uma **alternativa vetorizada** às estruturas `if/elif/else`
em loop. Ela avalia múltiplas condições de forma simultânea em todo o array,
sendo significativamente mais eficiente para datasets grandes.

### Faixas de Classificação de Receita

| Faixa | Condição | Label |
|---|---|---|
| Baixo | Receita < R$ 500 | Baixo Valor |
| Médio | R$ 500 ≤ Receita < R$ 5.000 | Médio Valor |
| Alto | Receita ≥ R$ 5.000 | Alto Valor |


In [ ]:
def criar_colunas_derivadas(df):
    """
    Cria colunas calculadas a partir do dataset limpo:
    - receita_total     : valor total da linha de venda (quantidade × preço)
    - mes / trimestre / ano : componentes extraídos da data
    - faixa_receita_item    : classificação do valor de cada venda (np.select)
    """
    df = df.copy()  # Cópia para preservar o DataFrame original

    # Receita total = quantidade × preço unitário (coluna calculada principal)
    df["receita_total"] = df["quantidade"] * df["preco_unitario"]

    # Extração de componentes temporais da coluna data_venda (já em formato datetime)
    df["mes"] = df["data_venda"].dt.month  # Mês numérico (1-12)
    # Trimestre formatado como 'Q1', 'Q2', 'Q3', 'Q4' — lambda formata o número
    df["trimestre"] = df["data_venda"].dt.quarter.apply(lambda q: f"Q{q}")
    df["ano"] = df["data_venda"].dt.year  # Ano (2024)

    # Classificação de receita usando np.select (alternativa vetorizada a if/elif/else)
    condicoes = [
        df["receita_total"] < 500,  # Baixo valor: menos de R$500
        (df["receita_total"] >= 500) & (df["receita_total"] < 5000),  # Médio: R$500-R$5.000
        df["receita_total"] >= 5000,  # Alto valor: R$5.000 ou mais
    ]
    rotulos = ["Baixo Valor", "Médio Valor", "Alto Valor"]  # Labels correspondentes
    # np.select avalia todas as condições de forma vetorizada (sem loop)
    df["faixa_receita_item"] = np.select(condicoes, rotulos, default="N/D")

    # Exibe as novas colunas criadas para verificação
    print("COLUNAS DERIVADAS CRIADAS")
    print(df[["data_venda", "receita_total", "mes", "trimestre", "faixa_receita_item"]].head())

    return df  # Retorna DataFrame com as 5 novas colunas adicionadas


# Aplica a criação de colunas derivadas na base v2 (sem outliers)
df = criar_colunas_derivadas(df_v2)
# Exibe as primeiras linhas das colunas derivadas para verificação
df[["data_venda", "receita_total", "mes", "trimestre", "faixa_receita_item"]].head()


## RF06 — Calcular Métricas Agregadas (groupby)

### O que é GroupBy?

O método `groupby` do pandas segue o paradigma **split-apply-combine**:
1. **Split**: divide os dados em grupos com base em uma ou mais colunas
2. **Apply**: aplica uma função de agregação (soma, média, contagem, etc.) a cada grupo
3. **Combine**: combina os resultados em um novo DataFrame

### 4 Dimensões Analisadas

| Dimensão | Métricas | Função |
|---|---|---|
| **Mês** | Receita total, quantidade vendida, número de vendas | `sum`, `count` |
| **Produto** | Receita total (top 5) | `sum` |
| **Categoria** | Receita total | `sum` |
| **Região** | Receita total, ticket médio | `sum`, `mean` |

### Uso do `.agg()` com Colunas Nomeadas

O método `.agg()` com **nomes de colunas explícitos** (ex: `receita_total=("receita_total", "sum")`)
produz um resultado mais claro e legível do que o uso de `agg(["sum"])`, pois cada
coluna de saída recebe um nome semântico que indica tanto o nome original quanto a
função aplicada.


In [ ]:
def calcular_metricas(df):
    """
    Calcula e retorna um dicionário com métricas agregadas por
    quatro dimensões: mês, produto, categoria e região.
    """
    metricas = {}  # Dicionário que armazenará todas as métricas agregadas

    # Métrica 1: Receita, quantidade e nº de vendas por mês (ordenado cronologicamente)
    metricas["por_mes"] = (
        df.groupby("mes")  # Agrupa por mês numérico (1-12)
        .agg(
            receita_total=("receita_total", "sum"),  # Soma da receita no mês
            quantidade=("quantidade", "sum"),  # Total de unidades vendidas
            n_vendas=("id_venda", "count"),  # Quantidade de transações
        )
        .reset_index()  # Converte o índice (mes) em coluna normal
        .sort_values("mes")  # Ordena por mês crescente
    )

    # Métrica 2: Top 5 produtos por receita total (ranking decrescente)
    metricas["top_produtos"] = (
        df.groupby("produto")["receita_total"]  # Agrupa por produto
        .sum()  # Soma a receita de cada produto
        .sort_values(ascending=False)  # Ordena do maior para o menor
        .head(5)  # Seleciona apenas os 5 primeiros (top 5)
        .reset_index()  # Converte para DataFrame com índice numérico
    )

    # Métrica 3: Receita total por categoria de vestuário
    metricas["por_categoria"] = (
        df.groupby("categoria")["receita_total"]  # Agrupa por categoria
        .sum()  # Soma a receita de cada categoria
        .reset_index()
        .sort_values("receita_total", ascending=False)  # Categoria com maior receita primeiro
    )

    # Métrica 4: Receita total e ticket médio por região geográfica
    metricas["por_regiao"] = (
        df.groupby("regiao")  # Agrupa por região (Sudeste, Sul, etc.)
        .agg(
            receita_total=("receita_total", "sum"),  # Soma da receita na região
            media_ticket=("receita_total", "mean"),  # Ticket médio por venda
        )
        .reset_index()
        .sort_values("receita_total", ascending=False)  # Região com maior receita primeiro
    )

    # Exibe todas as métricas calculadas no console
    for nome, tabela in metricas.items():
        print(f"\n=== {nome.upper().replace('_', ' ')} ===")
        print(tabela.to_string(index=False))  # to_string() remove o índice para legibilidade

    return metricas  # Retorna dicionário com 4 DataFrames de métricas


# Calcula todas as métricas agregadas e armazena no dicionário 'metricas'
metricas = calcular_metricas(df)


## RF07 — Segmentar Clientes por Nível de Gasto

| Gasto Total | Segmento |
|---|---|
| Abaixo de R$ 5.000 | Bronze |
| R$ 5.000 a R$ 15.000 | Prata |
| Acima de R$ 15.000 | Ouro |

### O que é Segmentação de Clientes?

Segmentação de clientes é o processo de **agrupar clientes com base em comportamentos
e padrões de compra semelhantes**. No nosso caso, classificamos clientes pelo valor
total gasto ao longo do período analisado.

### Por que segmentar?

- **Marketing direcionado**: campanhas personalizadas para cada perfil de cliente
- **Programas de fidelidade**: recompensas diferenciadas para clientes Ouro
- **Análise de receita**: entender a contribuição de cada segmento para o faturamento
- **Tomada de decisão**: a diretoria pode priorizar estratégias para diferentes segmentos


In [ ]:
def segmentar_clientes(df):
    """
    Agrupa os dados por cliente, calcula o total gasto por cada um
    e classifica em Bronze / Prata / Ouro usando uma função lambda
    com condicional encadeado (equivalente a if/elif/else).
    """
    # Agrupa por cliente e soma a receita total de cada um
    clientes_df = (
        df.groupby("cliente")["receita_total"]  # Agrupa por nome do cliente
        .sum()  # Soma todas as vendas do cliente no período
        .reset_index()  # Converte o índice em coluna normal
    )
    # Renomeia as colunas para nomes mais semânticos
    clientes_df.columns = ["cliente", "total_gasto"]

    # Lambda com operador ternário encadeado para classificar o segmento
    # Equivale a: if g > 15000: 'Ouro' elif g >= 5000: 'Prata' else: 'Bronze'
    clientes_df["segmento"] = clientes_df["total_gasto"].apply(
        lambda g: "Ouro" if g > 15000 else ("Prata" if g >= 5000 else "Bronze")
    )

    # Ordena por total gasto decrescente (clientes mais valiosos primeiro)
    clientes_df = clientes_df.sort_values("total_gasto", ascending=False)

    # Exibe top 10 clientes e distribuição de segmentos
    print("=== SEGMENTAÇÃO DE CLIENTES (Top 10) ===")
    print(clientes_df.head(10).to_string(index=False))
    # value_counts() mostra quantos clientes há em cada segmento
    print(f"\nDistribuição de segmentos:\n{clientes_df['segmento'].value_counts()}")

    return clientes_df  # Retorna DataFrame com cliente, total_gasto e segmento


# Segmenta os clientes e armazena no DataFrame 'clientes'
clientes = segmentar_clientes(df)
clientes.head()  # Exibe as primeiras linhas para verificação


## RF08 — Calcular Estatísticas com NumPy

Demonstra conversão para array, operações vetorizadas, broadcasting e boolean
indexing.

### Conceitos NumPy Demonstrados

1. **Conversão para Array** — `df["coluna"].to_numpy()` converte uma Série do Pandas
   em array NumPy, permitindo operações de baixo nível mais rápidas.

2. **Operações Vetorizadas** — Operações matemáticas sobre arrays inteiros sem laços
   explícitos. Funções utilizadas: `np.mean()`, `np.median()`, `np.std()`, `np.sum()`,
   `np.percentile()`, `np.sort()`.

3. **Broadcasting** — Operação escalar aplicada a cada elemento do array. Exemplo:
   `receitas / receitas.sum()` divide cada venda pelo total, resultando na
   participação percentual de cada venda no faturamento geral.

4. **Boolean Indexing** — Filtragem do array usando máscaras de True/False.
   Exemplo: `receitas > media` retorna um array booleano, e `.sum()` conta
   quantos valores atendem à condição (vendas acima da média).

### Funções NumPy Utilizadas

| Função | Propósito |
|---|---|
| `np.mean()` | Média aritmética |
| `np.median()` | Mediana (valor central) |
| `np.std()` | Desvio padrão |
| `np.sum()` | Soma total |
| `np.percentile()` | Percentis (P25, P75) |
| `np.sort()` | Ordenação |


In [ ]:
def calcular_estatisticas_numpy(df):
    """
    Usa NumPy diretamente sobre arrays para calcular estatísticas de receita.
    Demonstra: operações vetorizadas, broadcasting e boolean indexing.
    """
    # Converte a Série pandas para array NumPy — permite operações de baixo nível
    receitas = df["receita_total"].to_numpy()

    # Dicionário de estatísticas usando funções NumPy (operções vetorizadas)
    stats = {
        "media": float(np.mean(receitas)),  # Média aritmética de todas as receitas
        "mediana": float(np.median(receitas)),  # Valor central (50º percentil)
        "desvio_padrao": float(np.std(receitas)),  # Dispersão em relação à média
        "total": float(np.sum(receitas)),  # Soma total de todas as receitas
        "p25": float(np.percentile(receitas, 25)),  # 1º quartil (25% abaixo dele)
        "p75": float(np.percentile(receitas, 75)),  # 3º quartil (75% abaixo dele)
    }

    # Broadcasting: receitas / receitas.sum() divide cada venda pelo total
    # Resultado: participação percentual de cada venda no faturamento geral
    receitas_pct = (receitas / receitas.sum()) * 100
    # np.sort ordena o array; [-5:] pega os 5 maiores; round(2) arredonda
    print(f"  Participação das 5 maiores vendas no total: "
          f"{np.sort(receitas_pct)[-5:].round(2)}%")

    # Boolean indexing: receitas > media retorna array de True/False
    # .sum() conta True (vendas acima da média)
    acima_da_media = int((receitas > stats["media"]).sum())
    stats["acima_da_media"] = acima_da_media

    # Exibe todas as estatísticas formatadas
    print("\n=== ESTATÍSTICAS COM NUMPY ===")
    for k, v in stats.items():
        if k == "acima_da_media":
            print(f"  {k}: {v} vendas")  # Contagem (não monetário)
        else:
            print(f"  {k}: R$ {v:.2f}")  # Valores monetários com 2 casas decimais

    return stats  # Retorna dicionário com 7 estatísticas


# Calcula estatísticas com NumPy e armazena no dicionário 'stats'
stats = calcular_estatisticas_numpy(df)


## RF09 — Criar Visualizações com Matplotlib e Seaborn

### 3 Gráficos Gerados

| # | Tipo | Propósito | Biblioteca |
|---|---|---|---|
| 1 | **Gráfico de Linha** | Mostra a tendência de receita ao longo dos meses — identifica sazonalidade e meses de pico | Matplotlib |
| 2 | **Gráfico de Barras Horizontais** | Ranqueia os top 5 produtos por receita — identifica os mais vendidos | Seaborn |
| 3 | **Boxplot** | Mostra a distribuição de receita por região — identifica variabilidade e outliers regionais | Seaborn |

### Detalhes Técnicos

- `sns.set_theme(style="whitegrid", palette="muted")` aplica um tema profissional
  e consistente em todos os gráficos
- `plt.savefig(dpi=120)` exporta cada gráfico em PNG com resolução adequada para
  relatórios impressos e apresentações
- `plt.close()` libera memória após salvar cada gráfico, evitando acúmulo de figuras
  na sessão
- O eixo X do gráfico de linhas usa **abreviações de meses em português** (Jan, Fev,
  Mar, etc.) para melhor legibilidade


In [ ]:
def gerar_visualizacoes(df, metricas, output_dir="../outputs/graficos"):
    """
    Gera e exporta 3 gráficos informativos em PNG:
    1. Linha    — receita total por mês
    2. Barras   — top 5 produtos por receita
    3. Boxplot  — distribuição de receita por região
    """
    # Cria o diretório de saída para os gráficos
    os.makedirs(output_dir, exist_ok=True)
    # Aplica tema profissional do Seaborn com grid branco e paleta suave
    sns.set_theme(style="whitegrid", palette="muted")

    # Lista de abreviações de meses em português para o eixo X
    meses_abrev = ["Jan","Fev","Mar","Abr","Mai","Jun",
                   "Jul","Ago","Set","Out","Nov","Dez"]

    # --- Gráfico 1: Gráfico de Linha — Receita por Mês ---
    # Mostra tendência e sazonalidade ao longo do ano
    fig, ax = plt.subplots(figsize=(10, 5))  # Cria figura e eixo
    pm = metricas["por_mes"]  # Dados de receita mensal
    ax.plot(pm["mes"], pm["receita_total"], marker="o", linewidth=2)  # Linha com marcadores
    ax.set_title("Receita Total por Mês")
    ax.set_xlabel("Mês")
    ax.set_ylabel("Receita (R$)")
    ax.set_xticks(range(1, 13))  # Posições de 1 a 12 no eixo X
    ax.set_xticklabels(meses_abrev, rotation=45)  # Labels rotacionados para legibilidade
    fig.tight_layout()  # Ajusta layout para evitar cortes
    fig.savefig(f"{output_dir}/receita_por_mes.png", dpi=120)  # Salva em PNG com 120 DPI
    plt.show()  # Exibe o gráfico na célula
    plt.close()  # Libera memória da figura

    # --- Gráfico 2: Barras Horizontais — Top 5 Produtos ---
    # Barras horizontais são melhores para nomes longos de produtos
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=metricas["top_produtos"], y="produto", x="receita_total", ax=ax)
    ax.set_title("Top 5 Produtos por Receita Total")
    ax.set_xlabel("Receita Total (R$)")
    ax.set_ylabel("Produto")
    fig.tight_layout()
    fig.savefig(f"{output_dir}/top_produtos.png", dpi=120)
    plt.show()
    plt.close()  # Libera memória

    # --- Gráfico 3: Boxplot — Distribuição de Receita por Região ---
    # Boxplot mostra mediana, quartis e outliers visuais por região
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=df, x="regiao", y="receita_total", ax=ax)
    ax.set_title("Distribuição de Receita por Região")
    ax.set_xlabel("Região")
    ax.set_ylabel("Receita por Venda (R$)")
    plt.xticks(rotation=30)  # Rotaciona nomes das regiões para evitar sobreposição
    fig.tight_layout()
    fig.savefig(f"{output_dir}/dist_regiao.png", dpi=120)
    plt.show()
    plt.close()  # Libera memória da figura

    print(f"3 gráficos salvos em: {output_dir}")


# Gera e exporta os 3 gráficos do relatório
gerar_visualizacoes(df, metricas)


## RF10 — Organizar o Código em Funções Reutilizáveis

Todas as etapas acima já estão encapsuladas em funções com docstring, parâmetros e
retorno explícito (`limpar_dados`, `tratar_outliers`, `criar_colunas_derivadas`,
`calcular_metricas`, `segmentar_clientes`, `calcular_estatisticas_numpy`,
`gerar_visualizacoes`). A função `limpar_dados` foi reutilizada implicitamente como
base tanto para a v1 quanto para a v2 (via `tratar_outliers` sobre o resultado de
`limpar_dados`).

Abaixo, demonstramos uma função de ordem superior (callback) e o segundo uso
distinto de `lambda` no projeto.

### 7 Funções Reutilizáveis Criadas

| Função | Propósito |
|---|---|
| `limpar_strings_regex()` | Normaliza strings com regex (espaços extras) |
| `limpar_dados()` | Pipeline completo de limpeza em 4 etapas |
| `tratar_outliers()` | Detecta e trata outliers pelo método IQR |
| `criar_colunas_derivadas()` | Cria colunas calculadas (receita, mês, trimestre, faixa) |
| `calcular_metricas()` | Calcula métricas agregadas por 4 dimensões |
| `segmentar_clientes()` | Classifica clientes Bronze/Prata/Ouro |
| `calcular_estatisticas_numpy()` | Estatísticas com NumPy (vetorizadas) |
| `gerar_visualizacoes()` | Gera e exporta 3 gráficos em PNG |
| `exportar_resultados()` | Exporta CSV e JSON |
| `aplicar_transformacao()` | Função de ordem superior (callback) |

### Função de Ordem Superior (Callback)

Uma **função de ordem superior** é uma função que recebe outra função como
parâmetro. O padrão **callback** permite passar comportamentos específicos como
argumentos, promovendo **flexibilidade** e **reutilização de código**.

No exemplo abaixo, `aplicar_transformacao` recebe qualquer função (incluindo lambdas)
e a aplica a uma coluna do DataFrame, criando uma nova coluna transformada.


In [ ]:
def aplicar_transformacao(df, coluna, funcao):
    """
    Função de ordem superior: aplica qualquer função (incluindo lambdas)
    a uma coluna do DataFrame, criando uma coluna '_transformado'.

    Parâmetros:
        df     : DataFrame de entrada
        coluna : nome da coluna a transformar
        funcao : função (ou lambda) a aplicar — o 'callback'

    Retorna uma cópia do DataFrame com a nova coluna; não modifica o original.
    """
    df = df.copy()  # Cópia para preservar o DataFrame original
    # .apply(funcao) aplica a função callback a cada elemento da coluna
    df[f"{coluna}_transformado"] = df[coluna].apply(funcao)
    return df  # Retorna DataFrame com a nova coluna adicionada


# --- Uso 1: classificar vendas por ticket com lambda ---
# Lambda: classifica em 'Alto' se > R$5.000, senão 'Normal'
df_demo = aplicar_transformacao(
    df, "receita_total", lambda x: "Alto" if x > 5000 else "Normal"
)
print("=== EXEMPLO: classificação por ticket ===")
print(df_demo[["receita_total", "receita_total_transformado"]].head())

# --- Uso 2: arredondar receita em milhares com lambda ---
# Lambda: divide por 1000 e arredonda para 2 casas (ex: 99.9 → 0.10)
df_demo2 = aplicar_transformacao(
    df, "receita_total", lambda x: round(x / 1000, 2)
)
print("\n=== EXEMPLO: receita em milhares (R$ k) ===")
print(df_demo2[["receita_total", "receita_total_transformado"]].head())
# NOTA: df_demo e df_demo2 são apenas demonstrações — df principal não é alterado.


## RF11 — Ler e Escrever Arquivos (CSV e JSON)

### Dois Formatos de Arquivo Utilizados

| Formato | Uso no Projeto | Exemplo |
|---|---|---|
| **CSV** | Dados tabulares (métricas, segmentação) | `metricas_por_mes.csv`, `segmentacao_clientes.csv` |
| **JSON** | Dados estruturados (estatísticas) | `estatisticas_gerais.json` |

### Detalhes Técnicos de Exportação

- **Encoding `utf-8-sig`**: utilizado no CSV para garantir compatibilidade com o
  Microsoft Excel, que interpreta corretamente caracteres acentuados (ã, ç, é, etc.)
- **`ensure_ascii=False`**: utilizado no JSON para preservar caracteres acentuados
  em vez de convertê-los para sequências de escape Unicode
- **`json.dump(indent=2)`**: formata o JSON com indentação de 2 espaços para
  facilitar a leitura humana
- **Verificação com `json.load()`**: após exportar, o arquivo JSON é lido de volta
  para confirmar que foi gravado corretamente (teste de integridade)


In [ ]:
def exportar_resultados(metricas, clientes, stats):
    """
    Exporta os resultados da análise em dois formatos:
    - CSV : métricas mensais e segmentação de clientes
    - JSON: estatísticas gerais calculadas com NumPy

    Após exportar o JSON, faz a leitura de volta com json.load()
    para confirmar que o arquivo foi gravado corretamente.
    """
    # Cria o diretório de saída para relatórios
    os.makedirs("../outputs", exist_ok=True)

    # Exporta métricas mensais para CSV
    # encoding='utf-8-sig' garante compatibilidade com Excel (caracteres acentuados)
    metricas["por_mes"].to_csv(
        "../outputs/metricas_por_mes.csv", index=False, encoding="utf-8-sig"
    )
    print("CSV exportado: outputs/metricas_por_mes.csv")

    # Exporta segmentação de clientes para CSV
    clientes.to_csv(
        "../outputs/segmentacao_clientes.csv", index=False, encoding="utf-8-sig"
    )
    print("CSV exportado: outputs/segmentacao_clientes.csv")

    # Converte valores do stats para tipos serializáveis em JSON (float)
    stats_serializaveis = {k: round(float(v), 2) for k, v in stats.items()}
    caminho_json = "../outputs/estatisticas_gerais.json"
    # Exporta estatísticas para JSON com indentação e caracteres acentuados preservados
    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(stats_serializaveis, f, indent=2, ensure_ascii=False)
    print(f"JSON exportado: {caminho_json}")

    # Leitura de volta do JSON para verificar integridade do arquivo
    with open(caminho_json, encoding="utf-8") as f:
        lido = json.load(f)
    print("\nJSON lido de volta para confirmação:")
    print(json.dumps(lido, indent=2, ensure_ascii=False))


# Exporta todos os resultados: CSV (métricas, clientes) e JSON (estatísticas)
exportar_resultados(metricas, clientes, stats)


## RF12 — Consolidar a Análise e Salvar o Dataset Final

Optamos por **df_v2 (outliers removidos)** como base da análise final — ela reflete
melhor o comportamento típico de vendas para a reunião da diretoria. A v1 (com
outliers) continua disponível em `data/processed/v1_com_outliers/` para consulta.

### Por que escolher df_v2?

Outliers distorcem médias, totais e agregações. A versão v2 **preserva 93.8% dos
registros** originais (12.556 de 13.402 após limpeza) mas remove as anomalias que
poderiam levar a decisões equivocadas. Para a reunião da diretoria, dados
representativos do comportamento típico são mais úteis do que incluir exceções.

### Pipeline Completo: Dados Brutos → Dados Finais

```
data/raw/vendas.csv                    ← Dataset bruto com sujeira
        ↓
data/processed/v1_com_outliers/         ← Limpo, mas com outliers
        ↓
data/processed/v2_outliers_tratado/     ← Limpo e sem outliers
        ↓
data/final/vendas_final.csv             ← Base oficial com colunas derivadas
```

### Checklist de Arquivos Gerados

Ao final, um checklist verifica a existência de todos os 10 arquivos esperados:
- 4 datasets (raw, v1, v2, final)
- 2 relatórios CSV (métricas mensais, segmentação de clientes)
- 1 relatório JSON (estatísticas gerais)
- 3 gráficos PNG (receita por mês, top produtos, distribuição por região)


In [ ]:
# Cria o diretório para o dataset final
os.makedirs("../data/final", exist_ok=True)
# Salva o DataFrame final (df_v2 + colunas derivadas) em CSV
df.to_csv("../data/final/vendas_final.csv", index=False)
print("Dataset final salvo em: data/final/vendas_final.csv")

# Lista de todos os arquivos que o pipeline deve ter gerado
arquivos_esperados = [
    "../data/raw/vendas.csv",  # Dataset bruto original
    "../data/processed/v1_com_outliers/vendas_v1.csv",  # Limpo, com outliers
    "../data/processed/v2_outliers_tratado/vendas_v2.csv",  # Limpo, sem outliers
    "../data/final/vendas_final.csv",  # Dataset final com colunas derivadas
    "../outputs/metricas_por_mes.csv",  # Métricas mensais agregadas
    "../outputs/segmentacao_clientes.csv",  # Segmentação Bronze/Prata/Ouro
    "../outputs/estatisticas_gerais.json",  # Estatísticas NumPy em JSON
    "../outputs/graficos/receita_por_mes.png",  # Gráfico de linha
    "../outputs/graficos/top_produtos.png",  # Gráfico de barras
    "../outputs/graficos/dist_regiao.png",  # Boxplot por região
]

# Checklist: verifica se todos os arquivos foram criados com sucesso
print("\n=== CHECKLIST DE ARQUIVOS GERADOS ===")
for arq in arquivos_esperados:
    # os.path.exists() retorna True se o arquivo existe no disco
    status = "OK" if os.path.exists(arq) else "FALTANDO"
    print(f"  [{status}] {arq}")


## Conclusão

O pipeline DataView percorreu todo o ciclo de um Analista de Dados Júnior:
carga e inspeção do dataset bruto, limpeza (nulos, datas, strings), detecção e
tratamento de outliers em duas versões (v1/v2), criação de colunas derivadas,
agregações com `groupby`, segmentação de clientes, estatísticas vetorizadas com
NumPy, visualizações exportadas em PNG e relatórios finais em CSV/JSON.

### Próximos Passos Sugeridos

- **Testes automatizados**: adicionar testes unitários com `pytest` para as funções de
  limpeza, garantindo que mudanças futuras não quebrem o pipeline
- **Parametrização**: tornar configuráveis os limites de segmentação de clientes (Bronze/Prata/Ouro),
  faixas de receita e parâmetros do IQR
- **Dataset real**: conectar a dados reais do Kaggle ou dados.gov.br para validar a
  generalização das funções em cenários de produção
- **Dashboard interativo**: migrar as visualizações estáticas para um dashboard com
  Plotly, Streamlit ou Dash para exploração interativa
- **Documentação**: gerar um relatório automatizado em PDF/HTML com `nbconvert`
  para distribuição à diretoria
